In [ ]:
# -------------------------------
# Import libraries
# -------------------------------
import sys
# Replace this with your local path to the CausalICM folder
sys.path.append("/path/to/your/CausalICM/functions/")
print(sys.path)

import numpy as np
import pandas as pd
import copy
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge as ridge
import GPy

# Import CausalICM functions
from CausalICM.functions.data_simulation import data_simulation1
from CausalICM.functions.model_training import ICM

# -------------------------------
# Helper functions
# -------------------------------
rmse = lambda x,y: np.power(mean_squared_error(x,y), 0.5)

linear_reg_parameter_pairs = [
    (ridge(), {'alpha': np.logspace(-7, 7, 8), 'tol': [1e-3]})
]

def get_best_model_from_gridcv(X, Y, reg_parameter_pairs, print_flag=False, verbose=0):
    val_errs = []
    models = []
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
    
    for reg, parameters in reg_parameter_pairs:
        if print_flag:
            print('trying ', str(reg)[:50])
        model = GridSearchCV(reg, parameters, cv=2, refit=True, verbose=verbose, n_jobs=10)
        model.fit(X, Y)
        val_errs.append(rmse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model.best_estimator_))
    
    min_ind = val_errs.index(min(val_errs))
    if verbose != 0:
        print(str(models[min_ind]), 'val rmse:{}'.format(val_errs[min_ind]))
    return copy.deepcopy(models[min_ind])

def run_methodGP(X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, _internal_X_not_rct):
    # Assertions
    assert(X_rct.shape[0] == Y_rct.shape[0])
    assert(T_rct.shape[0] == Y_rct.shape[0])
    assert(X_obs.shape[0] == Y_obs.shape[0])
    assert(T_obs.shape[0] == Y_obs.shape[0])
    
    # Naive GP trained on observational data
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    f0_gp_model = GPy.models.GPRegression(np.vstack(X_obs[T_obs==0]), np.vstack(Y_obs[T_obs==0]), kern)
    f0_gp_model.optimize()
    
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    f1_gp_model = GPy.models.GPRegression(np.vstack(X_obs[T_obs==1]), np.vstack(Y_obs[T_obs==1]), kern)
    f1_gp_model.optimize()
    
    omega_rct_pred = f1_gp_model.predict(np.vstack(X_rct))[0] - f0_gp_model.predict(np.vstack(X_rct))[0]
    omega_obs_pred = f1_gp_model.predict(np.vstack(X_obs))[0] - f0_gp_model.predict(np.vstack(X_obs))[0]
    omega_not_rct_pred = f1_gp_model.predict(np.vstack(_internal_X_not_rct))[0] - f0_gp_model.predict(np.vstack(_internal_X_not_rct))[0]
    
    _mean_outcome = np.mean(Y_obs)
    
    def ite_adjusted_outcome(Y, T, _propensity, c=_mean_outcome):
        assert(0 < _propensity < 1)
        assert(len(Y.shape) == 1 and Y.shape == T.shape)
        _ite = np.zeros(Y.shape)
        _ite[T==1] = (Y[T==1] - c)/_propensity
        _ite[T==0] = -(Y[T==0] - c)/(1-_propensity)
        return _ite
    
    _propensity_rct = np.mean(T_rct)
    cate_rct_est = ite_adjusted_outcome(Y_rct, T_rct, _propensity=_propensity_rct)
    eta_rct_est = np.vstack(cate_rct_est) - np.vstack(omega_rct_pred)
    
    best_eta_est_linear = get_best_model_from_gridcv(X_rct.reshape(-1,1), eta_rct_est, linear_reg_parameter_pairs)
    
    return copy.deepcopy(best_eta_est_linear), omega_obs_pred, omega_not_rct_pred

# -------------------------------
# Initialize arrays
# -------------------------------
n_iterations = 100
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)

rmse_naiveGP_exp = np.zeros(n_iterations)
rmse_naiveGP_obs = np.zeros(n_iterations)
rmse_KallusGP = np.zeros(n_iterations)
rmse_ICM = np.zeros(n_iterations)

bias_naiveGP_exp = np.zeros(n_iterations)
bias_naiveGP_obs = np.zeros(n_iterations)
bias_KallusGP = np.zeros(n_iterations)
bias_ICM = np.zeros(n_iterations)

var_naiveGP_exp = np.zeros(n_iterations)
var_naiveGP_obs = np.zeros(n_iterations)
var_KallusGP = np.zeros(n_iterations)
var_ICM = np.zeros(n_iterations)

bias_naive_exp_df = pd.DataFrame(test_data)
bias_naive_obs_df = pd.DataFrame(test_data)
bias_kallus_df = pd.DataFrame(test_data)
bias_icm_df = pd.DataFrame(test_data)

all_data = []

# -------------------------------
# Load best rho
# -------------------------------
best_rho = pd.read_csv('sim1_best_rho.csv').values[0][0]

# -------------------------------
# Main loop
# -------------------------------
for i in range(n_iterations):
    print(f"Iteration {i+1}")
    data_exp, data_obs = data_simulation1(1000, beta0=-3.0, beta1=-3.0)
    
    df_exp = pd.DataFrame(data_exp, columns=["S", "X", "A", "Y"])
    df_exp["datasetNo"] = i + 1
    df_obs = pd.DataFrame(data_obs, columns=["S", "X", "A", "Y"])
    df_obs["datasetNo"] = i + 1
    combined_df = pd.concat([df_exp, df_obs], ignore_index=True)
    all_data.append(combined_df)
    
    # Split by treatment
    X0_obs, X1_obs = data_obs[:,1][data_obs[:,2]==0], data_obs[:,1][data_obs[:,2]==1]
    Y0_obs, Y1_obs = data_obs[:,3][data_obs[:,2]==0], data_obs[:,3][data_obs[:,2]==1]
    X0_exp, X1_exp = data_exp[:,1][data_exp[:,2]==0], data_exp[:,1][data_exp[:,2]==1]
    Y0_exp, Y1_exp = data_exp[:,3][data_exp[:,2]==0], data_exp[:,3][data_exp[:,2]==1]
    
    # True CATE
    trueCATE = 1 + test_data
    
    # Naive GP on observational data
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    m0_obs = GPy.models.GPRegression(np.vstack(X0_obs), np.vstack(Y0_obs), kern); m0_obs.optimize()
    m1_obs = GPy.models.GPRegression(np.vstack(X1_obs), np.vstack(Y1_obs), kern); m1_obs.optimize()
    naiveCATE_obs = m1_obs.predict(np.vstack(test_data))[0] - m0_obs.predict(np.vstack(test_data))[0]
    
    # Naive GP on experimental data
    m0_exp = GPy.models.GPRegression(np.vstack(X0_exp), np.vstack(Y0_exp), kern); m0_exp.optimize()
    m1_exp = GPy.models.GPRegression(np.vstack(X1_exp), np.vstack(Y1_exp), kern); m1_exp.optimize()
    naiveCATE_exp = m1_exp.predict(np.vstack(test_data))[0] - m0_exp.predict(np.vstack(test_data))[0]
    
    # Kallus GP
    X_all = np.r_[data_obs[:,1], data_exp[:,1]]
    Y_all = np.r_[data_obs[:,3], data_exp[:,3]]
    T_all = np.r_[data_obs[:,2], data_exp[:,2]]
    _mean_outcome = np.mean(data_obs[:,3])
    eta_2step_modelGP, omega_obs_pred, omega_eval_pred = run_methodGP(
        data_exp[:,1], data_exp[:,3], data_exp[:,2], 
        data_obs[:,1], data_obs[:,3], data_obs[:,2], 
        test_data
    )
    tau_pred_eval = eta_2step_modelGP.predict(test_data.reshape(-1,1)) + omega_eval_pred
    
    # ICM
    m0_icm, m1_icm = ICM(X_E=data_exp[:,1], X_O=data_obs[:,1], T_E=data_exp[:,2], T_O=data_obs[:,2], 
                         Y_E=data_exp[:,3], Y_O=data_obs[:,3], r=2, ID=1, AD=0, rho=best_rho)
    predCATE_exp_icm = m1_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0] - \
                       m0_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0]
    
    # Metrics
    rmse_naiveGP_exp[i] = mean_squared_error(naiveCATE_exp, trueCATE, squared=False)
    rmse_naiveGP_obs[i] = mean_squared_error(naiveCATE_obs, trueCATE, squared=False)
    rmse_KallusGP[i] = mean_squared_error(tau_pred_eval, trueCATE, squared=False)
    rmse_ICM[i] = mean_squared_error(predCATE_exp_icm, trueCATE, squared=False)
    
    bias_naiveGP_exp[i] = np.mean(naiveCATE_exp - trueCATE)
    bias_naiveGP_obs[i] = np.mean(naiveCATE_obs - trueCATE)
    bias_KallusGP[i] = np.mean(tau_pred_eval - trueCATE)
    bias_ICM[i] = np.mean(predCATE_exp_icm - trueCATE)
    
    var_naiveGP_exp[i] = np.var(naiveCATE_exp)
    var_naiveGP_obs[i] = np.var(naiveCATE_obs)
    var_KallusGP[i] = np.var(tau_pred_eval)
    var_ICM[i] = np.var(predCATE_exp_icm)
    
    bias_naive_exp_df[f"Iteration_{i+1}"] = np.vstack(naiveCATE_exp) - np.vstack(trueCATE)
    bias_naive_obs_df[f"Iteration_{i+1}"] = np.vstack(naiveCATE_obs) - np.vstack(trueCATE)
    bias_kallus_df[f"Iteration_{i+1}"] = np.vstack(tau_pred_eval) - np.vstack(trueCATE)
    bias_icm_df[f"Iteration_{i+1}"] = np.vstack(predCATE_exp_icm) - np.vstack(trueCATE)

# -------------------------------
# Save results
# -------------------------------
final_df_sim1 = pd.concat(all_data, ignore_index=True)
final_df_sim1.to_csv('datasets_sim1.csv', index=False)

sim1_modelComparison_RMSEs_df = pd.DataFrame(
    np.c_[rmse_naiveGP_exp, rmse_naiveGP_obs, rmse_KallusGP, rmse_ICM,
          bias_naiveGP_exp, bias_naiveGP_obs, bias_KallusGP, bias_ICM,
          var_naiveGP_exp, var_naiveGP_obs, var_KallusGP, var_ICM],
    columns=['RMSE_naiveGPexp', 'RMSE_naiveGPobs', 'RMSE_KallusGP', 'RMSE_ICM',
             'bias_naiveGPexp', 'bias_naiveGPobs', 'bias_KallusGP', 'bias_ICM',
             'var_naiveGPexp', 'var_naiveGPobs', 'var_KallusGP', 'var_ICM']
)

sim1_modelComparison_RMSEs_df.to_csv('sim1_modelComparison.csv', index=False)
bias_naive_exp_df.to_csv('bias_naive_exp.csv', index=False)
bias_naive_obs_df.to_csv('bias_naive_obs.csv', index=False)
bias_kallus_df.to_csv('bias_kallus.csv', index=False)
bias_icm_df.to_csv('bias_icm.csv', index=False)
